# Medallion Hospital
 
 #### Suelem Audelo
 #### 7/22/26
 #### source files: input/hospital_reference_master.xlsx and input/hospital_appointments_raw.csv
 #### Business objectives:

        1. Preserves the original source data.
        2. Creates an auditable Bronze layer.
        3. Cleans and validates records in Silver.
        4. Produces report-ready Gold tables for hospital operations and revenue analysis.



In [11]:
#imports
from pathlib import Path 
import shutil
from pyspark.sql import SparkSession




#Q3 define paths
base_path = Path.cwd().resolve()
input_path = base_path / "input"
raw_path = base_path / "raw"
bronze_path = base_path / "bronze"
silver_path = base_path / "silver"
gold_path = base_path / "gold"

raw_appt = str(raw_path / "hospital_appointments_raw.csv")
raw_ref = str(raw_path / "hospital_reference_master.xlsx")

In [12]:
shutil.copy(str(input_path / "hospital_appointments_raw.csv"), raw_appt)
shutil.copy(str(input_path / "hospital_reference_master.xlsx"),raw_ref)

print("Successfully copied files")

Successfully copied files


In [13]:
print(f"--- Files in {input_path.name} ---")

for item in input_path.iterdir():
    if item.is_file():
        file_stats = item.stat()
        print(f"file:{item.name}  extension: {item.suffix} size: {file_stats.st_size}")
        

print(f"--- Files in {raw_path.name} ---")
for item in raw_path.iterdir():
    if item.is_file():
        file_stats = item.stat()
        print(f"file:{item.name}  extension: {item.suffix} size: {file_stats.st_size}")
#matching extensions and size and name
    


--- Files in input ---
file:hospital_appointments_raw.csv  extension: .csv size: 8430
file:hospital_reference_master.xlsx  extension: .xlsx size: 30409
--- Files in raw ---
file:hospital_appointments_raw.csv  extension: .csv size: 8430
file:hospital_reference_master.xlsx  extension: .xlsx size: 30409


### The RAW layer should not contain business transformations. Why?
This is because the raw layer is used as a long term archive to avoid needing to pull from the source again in case of errors in the other pipeline layers.

In [ ]:
#Bronze layer
# Read hospital_appointments_raw.csv into a Spark DataFrame.

# Use options appropriate for a header-based CSV file.

# Initially load source columns in a way that prevents invalid values from being silently lost.

spark=(
        SparkSession.builder
        .appName("MedallionHospitalDataframe")
        .master("local[*]")
        .getOrCreate()
    )
raw_appt_df = spark.read.csv(raw_appt, header=True, inferSchema=False)
raw_ref_df = spark.read.csv(raw_ref, header=True, inferSchema=False)

#raw_ref is a xslx

raw_appt_df.take(2)
#raw_ref_df.take(2)

[Row(appointment_id='A0001', appointment_date='2026-02-05', patient_id='P1044', patient_name='Diya Sharma', age='32', gender='O', city='Pune', department='Cardiology', doctor_id='D101', doctor_name='Dr. Ananya Rao', appointment_status='COMPLETED', consultation_fee='650', discount_pct='0', amount_paid='650.0', payment_mode='UPI', phone_number='9131994523', source_system='WEB', ingestion_date='2026-04-01'),
 Row(appointment_id='A0002', appointment_date='2026-03-06', patient_id='P1042', patient_name='  Diya Sharma  ', age='78', gender='M', city='chennai', department='Orthopedics', doctor_id='D201', doctor_name='Dr. Karan Shah', appointment_status='completed', consultation_fee='500', discount_pct='15', amount_paid='0.0', payment_mode=None, phone_number='9732719211', source_system='MOBILE_APP', ingestion_date='2026-04-01')]

: 